# IntakeState Regression Tests
Tests for bug fixes made during Tier 1 policy integration.
Run from `CAPSTONE2026_SPRING/` root.

In [24]:
import importlib
import intake_engine.policies.complaint_policies
importlib.reload(intake_engine.policies.complaint_policies)

import intake_engine.policies.target_specs
import intake_engine.policies.base_complaint_policy
import intake_engine.policies.complaint_policies
import intake_engine.policies.policy_selector
import intake_engine.state.templates
import intake_engine.state.routing
import intake_engine.state.rule_based_parser
import intake_engine.llm_extractor
import intake_engine.IntakeState
import intake_engine.app_flow

importlib.reload(intake_engine.policies.target_specs)
importlib.reload(intake_engine.policies.base_complaint_policy)
importlib.reload(intake_engine.policies.complaint_policies)
importlib.reload(intake_engine.policies.policy_selector)
importlib.reload(intake_engine.state.templates)
importlib.reload(intake_engine.state.routing)
importlib.reload(intake_engine.state.rule_based_parser)
importlib.reload(intake_engine.llm_extractor)
importlib.reload(intake_engine.IntakeState)
importlib.reload(intake_engine.app_flow)

print('Reloaded.')

Reloaded.


In [25]:
from intake_engine.app_flow import IntakeAppFlow

PASS = '\033[92m PASS\033[0m'
FAIL = '\033[91m FAIL\033[0m'

results = []

def check(label, condition):
    status = PASS if condition else FAIL
    print(f'{status}  {label}')
    results.append(condition)
    return condition

print('Helpers ready.')

Helpers ready.


## 1. Chief complaint stored as string, not list

In [26]:
print('=== Chief complaint type ===')
app = IntakeAppFlow(conn=None)
app.new_session(session_name='complaint_type_test', auto_save=False)
app.start_intake(auto_save=False)
app.submit_answer('Headache', auto_save=False)

complaint = app.get_state()['chief_complaint_primary']
print(f'  chief_complaint_primary: {repr(complaint)}')
results.append(check('Chief complaint is a string', isinstance(complaint, str)))
results.append(check('Chief complaint is not empty', bool(complaint)))

=== Chief complaint type ===
  chief_complaint_primary: 'headache'
 PASS  Chief complaint is a string
 PASS  Chief complaint is not empty


## 2. Safety check loop fix

In [27]:
print('=== Safety check loop fix ===')
app = IntakeAppFlow(conn=None)
app.new_session(session_name='safety_check_loop_test', auto_save=False)
app.start_intake(auto_save=False)

# First set the complaint
app.submit_answer('Headache', auto_save=False)

# Answer critical followups quickly
for answer in ['No.', 'No.', 'No.', 'No.', 'No.', 'No.', 'No.']:
    app.submit_answer(answer, auto_save=False)

# Answer onset, duration, then give high severity to trigger safety check
app.submit_answer('This morning.', auto_save=False)   # onset
app.submit_answer('4 hours.', auto_save=False)        # duration
r = app.submit_answer('10 out of 10.', auto_save=False)  # severity — triggers severe_symptom_intensity
print(f'  After 10/10 severity: intent={r["next_intent"]}, flags={app.get_state()["flags"]}')
results.append(check(
    '10/10 severity triggers ask_immediate_safety_check',
    r['next_intent'] == 'ask_immediate_safety_check'
))

r2 = app.submit_answer('No, I am okay.', auto_save=False)
print(f'  After safety check answer: intent={r2["next_intent"]}, flags={app.get_state()["flags"]}')
results.append(check(
    'Safety check does not loop after answer',
    r2['next_intent'] != 'ask_immediate_safety_check'
))
results.append(check(
    'severe_symptom_intensity cleared after safety check',
    'severe_symptom_intensity' not in app.get_state()['flags']
))

=== Safety check loop fix ===
  After 10/10 severity: intent=ask_immediate_safety_check, flags=['severe_symptom_intensity']
 PASS  10/10 severity triggers ask_immediate_safety_check
  After safety check answer: intent=ask_timing, flags=[]
 PASS  Safety check does not loop after answer
 PASS  severe_symptom_intensity cleared after safety check


In [28]:
# Add this after the severity answer to debug
app.submit_answer('Headache', auto_save=False)
for answer in ['No.', 'No.', 'No.', 'No.', 'No.', 'No.', 'No.']:
    app.submit_answer(answer, auto_save=False)
app.submit_answer('This morning.', auto_save=False)
app.submit_answer('4 hours.', auto_save=False)
r = app.submit_answer('10 out of 10.', auto_save=False)

state = app.get_state()
print('severity stored as:', repr(state['hpi']['severity']))
print('flags:', state['flags'])
print('detect_basic_flags result:', app.state.detect_basic_flags())

severity stored as: '10/10'
flags: ['severe_symptom_intensity']
detect_basic_flags result: ['severe_symptom_intensity']


In [29]:
from intake_engine.IntakeState import IntakeState
state = IntakeState()
print(state._should_use_rule_based_first('ask_severity'))

True


In [30]:
import importlib
import intake_engine.state.rule_based_parser
importlib.reload(intake_engine.state.rule_based_parser)

from intake_engine.state.rule_based_parser import extract_severity
print(repr(extract_severity('10 out of 10.')))
print(repr(extract_severity('10 out of 10')))
print(repr(extract_severity('10/10')))

'10/10'
'10/10'
'10/10'


In [31]:
from intake_engine.state.rule_based_parser import extract_severity
import inspect
print(inspect.getsource(extract_severity))

def extract_severity(text):
    answer = text.strip().lower()

    import re
    for value in range(10, -1, -1):  # check higher numbers first
        if re.search(rf'\b{value}/10\b', answer):
            return f"{value}/10"
        if re.search(rf'\b{value} out of 10\b', answer):
            return f"{value}/10"

    tokens = answer.replace(",", " ").split()
    for token in tokens:
        if token.isdigit():
            numeric_value = int(token)
            if 0 <= numeric_value <= 10:
                return f"{numeric_value}/10"

    return text.strip()



## 3. policy_selector handles list complaint gracefully

In [32]:
print('=== policy_selector list input guard ===')
from intake_engine.policies.policy_selector import get_policy_for_complaint

try:
    policy = get_policy_for_complaint(['headache'])
    results.append(check('policy_selector handles list input without crash', True))
except Exception as e:
    results.append(check(f'policy_selector handles list input without crash — CRASHED: {e}', False))

try:
    policy = get_policy_for_complaint(None)
    results.append(check('policy_selector handles None input without crash', True))
except Exception as e:
    results.append(check(f'policy_selector handles None input — CRASHED: {e}', False))

=== policy_selector list input guard ===
 PASS  policy_selector handles list input without crash
 PASS  policy_selector handles None input without crash


## 4. Deterministic intents never hit LLM extractor

In [33]:
print('=== Deterministic intents use rule-based parser ===')
from intake_engine.IntakeState import IntakeState

state = IntakeState()
deterministic_intents = [
    'ask_main_reason_for_visit',
    'ask_immediate_safety_check',
    'recommend_immediate_attention',
    'end_intake_early',
    'end_intake_naturally',
    'summarize_and_check_for_anything_else',
]
for intent in deterministic_intents:
    results.append(check(
        f'{intent} is deterministic',
        state._is_deterministic_intent(intent)
    ))

=== Deterministic intents use rule-based parser ===
 PASS  ask_main_reason_for_visit is deterministic
 PASS  ask_immediate_safety_check is deterministic
 PASS  recommend_immediate_attention is deterministic
 PASS  end_intake_early is deterministic
 PASS  end_intake_naturally is deterministic
 PASS  summarize_and_check_for_anything_else is deterministic


## 5. Full headache intake does not loop

In [34]:
print('=== Full headache intake — no loops ===')
app = IntakeAppFlow(conn=None)
app.new_session(session_name='headache_no_loop', auto_save=False)
app.start_intake(auto_save=False)

answers = [
    'I have a headache.',
    'No.',                                                  # sudden_severe_onset
    'No weakness, numbness, or trouble speaking.',          # neurologic_symptoms
    'No vision changes.',                                   # visual_changes
    'No confusion.',                                        # confusion_or_ams
    'No fever or neck stiffness.',                          # fever_or_neck_stiffness
    'No head trauma.',                                      # head_trauma
    'No, not pregnant.',                                    # pregnancy_or_postpartum_context
    'This morning.',                                        # onset
    'About four hours.',                                    # duration
    '7 out of 10.',                                         # severity
    'It comes and goes.',                                   # timing
    'About the same.',                                      # course
    'Front of my head.',                                    # location
    'Throbbing.',                                           # character
    'Bright light makes it worse.',                         # aggravating_factors
    'Rest helps a little.',                                 # relieving_factors
    'Nausea.',                                              # associated_symptoms
    'No medications.',                                      # medications
    'No allergies.',                                        # allergies
    'No, light sensitivity.',                               # photophobia
    'No sound sensitivity.',                                # phonophobia
    'Yes, nausea.',                                         # nausea_or_vomiting
    'No warning symptoms before.',                          # aura_features
    'No, exertion did not trigger it.',                     # exertional_trigger
    'No change with position.',                             # positional_component
    'No, this is my usual headache pattern.',               # new_or_progressive_pattern
    'No, I have not been overusing pain medicine.',         # medication_overuse_context
]

seen_intents = []
for answer in answers:
    r = app.submit_answer(answer, auto_save=False)
    intent = r['next_intent']
    seen_intents.append(intent)

no_consecutive_loops = all(
    seen_intents[i] != seen_intents[i-1]
    for i in range(1, len(seen_intents))
)
results.append(check('No consecutive repeated intents', no_consecutive_loops))
results.append(check(
    'Intake reaches wrap-up or completion',
    any(i in seen_intents for i in ['end_intake_naturally', 'summarize_and_check_for_anything_else'])
))

state = app.get_state()
results.append(check('HPI onset captured', bool(state['hpi']['onset'])))
results.append(check('HPI severity captured', bool(state['hpi']['severity'])))
results.append(check('neurologic_symptoms answered', state['policy_answers']['neurologic_symptoms'] is not None))

print('\nIntent sequence:')
for i, intent in enumerate(seen_intents):
    print(f'  {i+1:>2}. {intent}')

=== Full headache intake — no loops ===
 PASS  No consecutive repeated intents
 PASS  Intake reaches wrap-up or completion
 PASS  HPI onset captured
 PASS  HPI severity captured
 PASS  neurologic_symptoms answered

Intent sequence:
   1. ask_sudden_severe_onset
   2. ask_neurologic_symptoms
   3. ask_visual_changes
   4. ask_confusion_or_ams
   5. ask_fever_or_neck_stiffness
   6. ask_head_trauma
   7. ask_pregnancy_or_postpartum_context
   8. ask_onset
   9. ask_duration
  10. ask_severity
  11. ask_timing
  12. ask_course
  13. ask_location
  14. ask_character
  15. ask_aggravating_factors
  16. ask_relieving_factors
  17. ask_associated_symptoms
  18. ask_medications
  19. ask_allergies
  20. ask_photophobia
  21. ask_phonophobia
  22. ask_nausea_or_vomiting
  23. ask_aura_features
  24. ask_exertional_trigger
  25. ask_positional_component
  26. ask_new_or_progressive_pattern
  27. ask_medication_overuse_context
  28. summarize_and_check_for_anything_else


## 6. New policy complaint routing works end-to-end

In [35]:
print('=== New policy routing via app flow ===')
new_complaints = [
    ('nausea',           'nausea_vomiting'),
    ('i fainted',        'syncope'),
    ('heart racing',     'palpitations'),
    ('i had a seizure',  'seizure'),
    ('my ear hurts',     'ear_pain'),
    ('my eye hurts',     'eye_pain'),
    ('i have a rash',    'rash'),
    ('fever',            'fever'),
]
for complaint, expected_policy in new_complaints:
    app = IntakeAppFlow(conn=None)
    app.new_session(session_name=f'route_test_{expected_policy}', auto_save=False)
    app.start_intake(auto_save=False)
    r = app.submit_answer(complaint, auto_save=False)
    stored = app.get_state()['chief_complaint_primary']
    next_intent = r['next_intent']
    results.append(check(
        f'"{complaint}" -> stored as string, routes to policy intent',
        isinstance(stored, str) and next_intent not in {None, 'ask_main_reason_for_visit'}
    ))
    print(f'    complaint={repr(stored)}, next_intent={next_intent}')

=== New policy routing via app flow ===
 PASS  "nausea" -> stored as string, routes to policy intent
    complaint='nausea', next_intent=ask_onset
 PASS  "i fainted" -> stored as string, routes to policy intent
    complaint='i fainted', next_intent=ask_onset
 PASS  "heart racing" -> stored as string, routes to policy intent
    complaint='heart racing', next_intent=ask_onset
 PASS  "i had a seizure" -> stored as string, routes to policy intent
    complaint='i had a seizure', next_intent=ask_onset
 PASS  "my ear hurts" -> stored as string, routes to policy intent
    complaint='ear hurts', next_intent=ask_onset
 PASS  "my eye hurts" -> stored as string, routes to policy intent
    complaint='eye hurts', next_intent=ask_onset
 PASS  "i have a rash" -> stored as string, routes to policy intent
    complaint='rash', next_intent=ask_onset
 PASS  "fever" -> stored as string, routes to policy intent
    complaint='fever', next_intent=ask_onset


In [36]:
import inspect
from intake_engine.IntakeState import IntakeState
src = inspect.getsource(IntakeState.process_answer_with_llm)
print('rule_based_deterministic' in src)

True


## Summary

In [37]:
total = len(results)
passed = sum(results)
failed = total - passed
print(f'\n==============================')
print(f'  Results: {passed}/{total} passed')
if failed:
    print(f'  FAILED:  {failed}')
else:
    print(f'  All tests passed!')
print(f'==============================')


  Results: 52/52 passed
  All tests passed!
